# ER Diagram

<!-- <img src="er_diagram.png" alt="ER diagram" width="400" height="200">
<p align="center"><img src="image.png" alt="Centered image"></p> -->

The ER diagram does not include everything, it includes the most common subset used. Full description at https://aisynphys.readthedocs.io/en/current-release/api_schema.html#api-schema

![ER Diagram](er_diagram.png "ER diagram")

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import sqlite3
import pandas as pd
import pyqtgraph
print(pyqtgraph.__file__)
import neuroanalysis
print(neuroanalysis.__file__)
import neuroanalysis.util
import aisynphys
import aisynphys.database
print(aisynphys.__file__)

from IPython.display import display, Markdown

c:\Users\snrambo\AppData\Local\miniconda3\envs\aisynphys39\lib\site-packages\pyqtgraph\__init__.py
C:\Users\snrambo\neuroanalysis\neuroanalysis\__init__.py
C:\Users\snrambo\aisynphys\aisynphys\__init__.py


# Set up DB connection

To get this working, I have:
* Downloaded aisynphys source code
* Downloaded neuroanalysis source code
* Set up environment in anaconda, with python 3.9:

```powershell
# clone repo aisynphys
# change to python 3.9 in yaml
mamba create -n aisynphys39 python=3.9
conda activate aisynphys39
mamba install numpy scipy pandas matplotlib sqlalchemy pyqt pyqtgraph numba h5py -c conda-forge
pip install lmfit
cd C:\Users\myuser\src\aisynphys
python setup.py develop
# cd to user
git clone https://github.com/alleninstitute/neuroanalysis
cd C:\Users\myuser\neuroanalysis
python setup.py develop
# Then was a problem with missing pyyaml, so downloaded that
mamba install -c conda-forge pyyaml
```

In [2]:
from aisynphys.database import SynphysDatabase # https://aisynphys.readthedocs.io/en/current-release/database_access.html#database-access
from pathlib import Path

# Define cache directory
CACHE_PATH = Path("data/aisynphys/cache").resolve()

# Create a cache directory if it does not exist
CACHE_PATH.mkdir(parents=True, exist_ok=True)

# Tell AISynPhys to use this cache
aisynphys.config.cache_path = CACHE_PATH
# SynphysDatabase.set_cache_path(CACHE_PATH)


# Make cache
DATA_ROOT = Path("data/aisynphys/cache")

# Inspect available database versions
SynphysDatabase.list_versions()  # List all available versions

# Load small database -- Uncomment to load
# DB_VERSION = 'synphys_r2.1_medium.sqlite'
# db = SynphysDatabase.load_version(DB_VERSION)

[{'db_file': 'synphys_r1.0_2019-08-29_small.sqlite',
  'url': 'https://allen-synphys.s3-us-west-2.amazonaws.com/synphys_r1.0_small.sqlite',
  'schema_version': '15',
  'release_version': '1.0',
  'db_size': 'small'},
 {'db_file': 'synphys_r1.0_small.sqlite',
  'url': 'https://allen-synphys.s3-us-west-2.amazonaws.com/synphys_r1.0_small.sqlite',
  'schema_version': '15',
  'release_version': '1.0',
  'db_size': 'small'},
 {'db_file': 'synphys_r1.0_2019-08-29_medium.sqlite',
  'url': 'https://allen-synphys.s3-us-west-2.amazonaws.com/synphys_r1.0_medium.sqlite',
  'schema_version': '15',
  'release_version': '1.0',
  'db_size': 'medium'},
 {'db_file': 'synphys_r1.0_medium.sqlite',
  'url': 'https://allen-synphys.s3-us-west-2.amazonaws.com/synphys_r1.0_medium.sqlite',
  'schema_version': '15',
  'release_version': '1.0',
  'db_size': 'medium'},
 {'db_file': 'synphys_r1.0_2019-08-29_full.sqlite',
  'url': 'https://allen-synphys.s3-us-west-2.amazonaws.com/synphys_r1.0_full.sqlite',
  'schema_

In [3]:
# Get file path and ensure it exist
DB_PATH = Path('data/aisynphys/cache/database/synphys_r2.1_medium.sqlite').resolve()  # database

assert DB_PATH.exists(), f"Database file not found: {DB_PATH}"

In [4]:
# Create SQLite connection to SQLite database

con = sqlite3.connect(DB_PATH)  # Create a connection object that "connects to the database"
cur = con.cursor()  # database cursor

# Table overview

In [5]:
table_list = cur.execute(
    "SELECT name FROM sqlite_master WHERE type='table'"
).fetchall()

for i, name in enumerate(table_list):
    print(i, name[0])


# Because they are stored in the form of ('pair',), accessing them by (str(list[i])[2:-3]) displays from the third index which is the first letter (excluding "('" ), and excluding the last "',)"
print(f"{str(table_list[9])[2:-3]}\n{str(table_list[20])[2:-3]}")

0 metadata
1 pipeline
2 slice
3 experiment
4 electrode
5 sync_rec
6 cortical_site
7 cell
8 recording
9 pair
10 intrinsic
11 morphology
12 test_pulse
13 stim_pulse
14 baseline
15 cortical_cell_location
16 patch_seq
17 patch_clamp_recording
18 stim_spike
19 pulse_response
20 synapse
21 poly_synapse
22 dynamics
23 synapse_prediction
24 gap_junction
25 synapse_model
26 multi_patch_probe
27 avg_response_fit
28 pulse_response_fit
29 pulse_response_strength
30 resting_state_fit
31 conductance
pair
synapse


# Explore tables

In [6]:
query = """
            SELECT DISTINCT qual_morpho_type FROM morphology
        """

df = pd.read_sql_query(query, con)

df

,qual_morpho_type
0,None
1,L2/3 IT cell
2,L5 IT
3,L6a non-tuffed/simple tuft
4,Martinotti Cell
5,Martinotti cell
6,"Tufted L4,5"
7,bipolar
8,bipolar cell
9,bipolar cell?


In [ ]:
query = """
            SELECT DISTINCT target_region FROM experiment
        """

df = pd.read_sql_query(query, con)

df

,target_region
0,VisP
1,ALM
2,TCx
3,FCx
4,None
5,
6,PCx
7,TEa
8,ACC
9,OCx


In [37]:
query = """
            SELECT DISTINCT species FROM slice
        """

df = pd.read_sql_query(query, con)

df

,species
0,mouse
1,human


In [ ]:
query = """
            SELECT c.cell_class FROM cell INNER JOIN
        """

df = pd.read_sql_query(query, con)

df

,id,experiment_id,ext_id,electrode_id,cre_type,target_layer,position,depth,cell_class,cell_class_nonsynaptic,meta
0,1,1,1,1,unknown,,null,NaN,None,None,"{""lims_specimen_id"": null, ""transgenic_cell_cl..."
1,2,1,2,2,unknown,,null,NaN,None,None,"{""lims_specimen_id"": null, ""transgenic_cell_cl..."
2,3,2,2,9,unknown,,null,NaN,None,None,"{""lims_specimen_id"": null, ""transgenic_cell_cl..."
3,4,1,3,3,unknown,,null,NaN,None,None,"{""lims_specimen_id"": null, ""transgenic_cell_cl..."
4,5,1,4,4,unknown,,null,NaN,None,None,"{""lims_specimen_id"": null, ""transgenic_cell_cl..."
...,...,...,...,...,...,...,...,...,...,...,...
24734,24844,5268,3,40687,pvalb,4,"[-0.0035553204560413235, 0.0005873930501915023...",0.000079,in,in,"{""lims_specimen_id"": 1147725593, ""transgenic_c..."
24735,24845,5269,1,40693,unknown,4,"[-0.003751572616205674, 0.0005186194818918567,...",0.000063,ex,ex,"{""lims_specimen_id"": 1146816821, ""transgenic_c..."
24736,24846,5269,3,40695,unknown,4,"[-0.0036679687909781933, 0.0004364560591056943...",0.000099,None,None,"{""lims_specimen_id"": 1146816814, ""transgenic_c..."
24737,24847,5269,6,40698,unknown,4,"[-0.003778551854032129, 0.0004735732325434664,...",0.000060,ex,ex,"{""lims_specimen_id"": 1146816827, ""transgenic_c..."



VisP: Primary Visual Cortex
ALM: Anterior Lateral Motor Cortex
TCx: Temporal Cortex
FCx: Frontal Cortex
None
  (empty):
PCx: Piriform Cortex
TEa: Temporal Association Area
ACC: Anterior Cingulate Cortex
OCx: Occipital Cortex

TODO
* Clean data: Only include relevant 
    * Mouse
    * Target region_ VisP
    * Pairs in this


# Querying 

In [83]:
df = pd.read_sql_query(
    # "SELECT COUNT(DISTINCT id) FROM pair;"
    "SELECT * FROM pair"
    , con)
df

,id,experiment_id,pre_cell_id,post_cell_id,has_synapse,has_polysynapse,has_electrical,crosstalk_artifact,n_ex_test_spikes,n_in_test_spikes,distance,lateral_distance,vertical_distance,reciprocal_id,meta
0,1,1,1,2,NaN,NaN,NaN,None,0,0,NaN,NaN,NaN,17,null
1,2,2,3,7,NaN,NaN,NaN,None,0,0,NaN,NaN,NaN,13,null
2,3,2,3,9,NaN,NaN,NaN,None,0,0,NaN,NaN,NaN,24,null
3,4,1,1,4,NaN,NaN,NaN,None,0,0,NaN,NaN,NaN,32,null
4,5,2,3,11,NaN,NaN,NaN,None,0,0,NaN,NaN,NaN,35,null
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
123501,123808,5269,24847,24846,0.0,0.0,0.0,None,0,0,0.000123,0.000083,0.000058,123805,null
123502,123809,5269,24847,24848,0.0,0.0,0.0,None,0,0,0.000035,0.000012,0.000028,123812,null
123503,123810,5269,24848,24845,0.0,0.0,0.0,None,0,0,0.000026,0.000022,0.000003,123803,null
123504,123811,5269,24848,24846,0.0,0.0,0.0,None,0,0,0.000133,0.000072,0.000086,123806,null


In [30]:
df = pd.read_sql_query("SELECT * FROM cell LIMIT 4", con)
df

,id,experiment_id,ext_id,electrode_id,cre_type,target_layer,position,depth,cell_class,cell_class_nonsynaptic,meta
0,1,1,1,1,unknown,,null,None,None,None,"{""lims_specimen_id"": null, ""transgenic_cell_cl..."
1,2,1,2,2,unknown,,null,None,None,None,"{""lims_specimen_id"": null, ""transgenic_cell_cl..."
2,3,2,2,9,unknown,,null,None,None,None,"{""lims_specimen_id"": null, ""transgenic_cell_cl..."
3,4,1,3,3,unknown,,null,None,None,None,"{""lims_specimen_id"": null, ""transgenic_cell_cl..."


In [27]:
# df_slice = pd.read_sql_query("SELECT mouse_id FROM slice  WHERE species = 'mouse'", con)  # IDs of mouse
# df_experiment_ids = pd.read_sql_query("SELECT id AS exp_id FROM experiment WHERE target_region = 'VisP'", con)  # IDs of experiments in Primary Visual Cortex
# df_cell = pd.read_sql_query("SELECT * FROM cell", con)
# df_pair = pd.read_sql_query("SELECT * FROM pair", con)

df = pd.read_sql_query("SELECT DISTINCT cell_class from cell", con)
df

,cell_class
0,None
1,ex
2,in
3,mixed


We want all cell pairs that:

* Belong to an experiment
* Belong to a mouse slice
* Experiment targets VisP
* Both cells exist
* Both cell classes are known
* Presynaptic cell is not mixed

In [46]:
# SQL query to retrieve the desired info

query = """ 
            SELECT 
                --e.id AS experiment_id, 
                p.experiment_id,
                p.pre_cell_id, 
                p.post_cell_id, 
                p.has_synapse, 
                pre.cell_class AS pre_cell_class,
                post.cell_class AS post_cell_class

            FROM pair p

            -- Join on both pre and post synaptic cell
            INNER JOIN cell pre ON p.pre_cell_id = pre.id
            INNER JOIN cell post ON p.post_cell_id = post.id

            -- Join with experiment and slice
            INNER JOIN experiment e ON e.id = p.experiment_id
            INNER JOIN slice s ON e.slice_id = s.id

            -- Filter out what is relevant for us
            WHERE 
                s.species = 'mouse' 
                AND e.target_region = 'VisP' 

                -- No null values in synapse and cell classes
                AND p.has_synapse IS NOT NULL
                AND pre.cell_class IS NOT NULL
                AND post.cell_class IS NOT NULL

                -- We allow mixed cell class in postsynaptic cells, but not in presynaptic cell class
                AND pre.cell_class != 'mixed'; 
        """

df_my_experiment = pd.read_sql_query(query, con)

df_my_experiment


,experiment_id,pre_cell_id,post_cell_id,has_synapse,pre_cell_class,post_cell_class
0,25,158,159,0,in,in
1,25,158,161,0,in,in
2,25,158,163,0,in,in
3,25,159,158,0,in,in
4,25,159,161,0,in,in
...,...,...,...,...,...,...
37116,5269,24845,24848,1,ex,in
37117,5269,24847,24845,0,ex,ex
37118,5269,24847,24848,0,ex,in
37119,5269,24848,24845,0,in,ex


## Aggregating to see counts of synapses

Querying using aggregation

In [52]:
# SQL query to retrieve the desired info

query = """ 
            SELECT 
                p.experiment_id,
                COUNT(*) AS total_pairs,
                SUM(p.has_synapse) AS num_synapses,
                COUNT(*) - SUM(p.has_synapse) AS num_no_synapses


            FROM pair p

            INNER JOIN cell pre ON p.pre_cell_id = pre.id
            INNER JOIN cell post ON p.post_cell_id = post.id
            INNER JOIN experiment e ON e.id = p.experiment_id
            INNER JOIN slice s ON e.slice_id = s.id


            WHERE 
                s.species = 'mouse' 
                AND e.target_region = 'VisP' 

                -- No null values in synapse and cell classes
                AND p.has_synapse IS NOT NULL
                AND pre.cell_class IS NOT NULL
                AND post.cell_class IS NOT NULL

                -- We allow mixed cell class in postsynaptic cells, but not in presynaptic cell class
                AND pre.cell_class != 'mixed'
                
            GROUP BY p.experiment_id;
        """

df_synapses = pd.read_sql_query(query, con)

df_synapses


,experiment_id,total_pairs,num_synapses,num_no_synapses
0,25,12,0,12
1,35,2,0,2
2,36,9,0,9
3,38,6,0,6
4,39,20,0,20
...,...,...,...,...
2436,5265,6,4,2
2437,5266,1,1,0
2438,5267,2,0,2
2439,5268,2,1,1


In [55]:
# How many in total
total_synapses = df_synapses["num_synapses"].sum()
total_not_synapses = df_synapses["num_no_synapses"].sum()
print(f"Total synapses: {total_synapses}")
print(f"Total not synapses: {total_not_synapses}")

Total synapses: 2440
Total not synapses: 34681


To look for where it is above some number of synapses we can use having:

In [57]:
# SQL query to retrieve the desired info

query = """ 
            SELECT 
                p.experiment_id,
                COUNT(*) AS total_pairs,
                SUM(p.has_synapse) AS num_synapses,
                COUNT(*) - SUM(p.has_synapse) AS num_no_synapses


            FROM pair p

            INNER JOIN cell pre ON p.pre_cell_id = pre.id
            INNER JOIN cell post ON p.post_cell_id = post.id
            INNER JOIN experiment e ON e.id = p.experiment_id
            INNER JOIN slice s ON e.slice_id = s.id


            WHERE 
                s.species = 'mouse' 
                AND e.target_region = 'VisP' 

                -- No null values in synapse and cell classes
                AND p.has_synapse IS NOT NULL
                AND pre.cell_class IS NOT NULL
                AND post.cell_class IS NOT NULL

                -- We allow mixed cell class in postsynaptic cells, but not in presynaptic cell class
                AND pre.cell_class != 'mixed'
                
            GROUP BY p.experiment_id
            HAVING num_synapses > 10;
        """

df_synapses = pd.read_sql_query(query, con)

df_synapses


,experiment_id,total_pairs,num_synapses,num_no_synapses
0,1504,42,11,31
1,1864,30,11,19
2,2771,30,11,19
3,2808,20,12,8
4,3172,56,19,37
5,3228,30,12,18
6,3262,42,11,31


#### Debugging row differences with the other version

**Subtract differences too see why I get less rows**

In [ ]:
df_slice = pd.read_sql_query("SELECT * from "+ str(table_list[2])[2:-3], con)
df_experiment = pd.read_sql_query("SELECT * from "+ str(table_list[3])[2:-3], con)
df_cell = pd.read_sql_query("SELECT * from "+ str(table_list[7])[2:-3], con)
df_pair = pd.read_sql_query("SELECT * from "+ str(table_list[9])[2:-3], con)


# Obtener los experimentos en df2 que son de ratón
######## Get the experiments in df_2 = df_slice that are mouse
mouse_ids = df_slice[df_slice['species'] == 'mouse']['id']

# Obtener los experimentos en df3 que tienen target_region == VISp
######## Get the experiments in df_3 = df_experiment that have target_region == VISp
visp_ids = df_experiment[df_experiment['target_region'] == 'VisP']['id']
print(visp_ids)

# Intersección de ambos conjuntos de experimentos
######## Intersection of both sets of experiments
valid_exp_ids = set(mouse_ids).intersection(set(visp_ids))

# Filtrar df9 con esos experimentos válidos
######## Filter df9 = df_pair with those valid experiments
df_pair_2 = df_pair[df_pair['experiment_id'].isin(mouse_ids)]


df_merged = pd.merge(df_pair[['id', 'experiment_id', 'has_synapse', 'pre_cell_id', 'post_cell_id']],
                     df_cell[['id', 'cell_class']],
                     left_on='pre_cell_id', right_on='id')
df_merged.rename(columns={'cell_class': 'pre_cell_class'}, inplace=True)

# Seleccionamos y ordenamos las columnas deseadas
df_result = df_merged[['experiment_id','pre_cell_id', 'post_cell_id', 'has_synapse', 'pre_cell_class']]
df_result = df_result[df_result['pre_cell_class']!= 'mixed']

# Merge adicional entre post_cell_id y id en df7
df_result = pd.merge(df_result,
                     df_cell[['id', 'cell_class']],
                     left_on='post_cell_id', right_on='id',
                     suffixes=('_pre', '_post'))

# Filtramos los resultados para que los 'pre_cell_class' no sean 'mixed'
df_result = df_result[df_result['pre_cell_class'] != 'mixed']

# Si quieres cambiar el nombre de la columna 'cell_class_post' por ejemplo, puedes hacerlo de la siguiente manera:
df_result.rename(columns={'cell_class': 'post_cell_class'}, inplace=True)


df_result = df_result[['experiment_id','pre_cell_id', 'post_cell_id', 'has_synapse', 'pre_cell_class', 'post_cell_class']]
df_clean = df_result.dropna(subset=['pre_cell_class', 'post_cell_class', 'has_synapse'])

# display(df_clean)

df_check = df_clean.merge(
    df_experiment[['id', 'target_region']],
    left_on='experiment_id',
    right_on='id',
    how='left'
)

print("checking")
df_check['target_region'].value_counts()

#### TEST: Other regions were included

df_check = df_clean.merge(
    df_experiment[['id', 'target_region']],
    left_on='experiment_id',
    right_on='id',
    how='left'
)

print("checking")
df_check['target_region'].value_counts()

0          1
1          2
2          3
3          4
4          5
        ... 
5053    5265
5054    5266
5055    5267
5056    5268
5057    5269
Name: id, Length: 4369, dtype: int64
checking


target_region
VisP    37127
TCx      2573
          213
FCx       166
ALM        90
PCx        85
TEa        30
Name: count, dtype: int64